# Architektura Aplikacji w Pythonie — Zestaw Zaliczeniowy

**WSEI Kraków · semestr letni 2026 · prowadzący: Michał Madejski**

---

## Filozofia tego zestawu

Sześć laboratoriów dało Ci sześć narzędzi. Ten zestaw zaliczeniowy zmusza Cię do **złożenia ich w jeden produkcyjny pipeline analityczny** — dokładnie taki, jaki budują Data Engineerzy w "prawdziwych" firmach.

**Wspólny dataset:** [`stanfordnlp/imdb`](https://huggingface.co/datasets/stanfordnlp/imdb) z Hugging Face Hub — 50 000 recenzji filmów z etykietami sentymentu (pozytywna / negatywna).

**Reguły:**
1. Każdy lab ma blok: **Teoria → Przykład rozwiązany → Zadanie samodzielne**.
2. Zadania samodzielne **rozszerzają** przykład — dokładnie ten sam pattern, inny scenariusz.
3. Cały notebook ma być **uruchamialny od góry do dołu**. Brak hardkodowanych ścieżek, brak ręcznych downloadów.
4. Kod ma być **czytelny**: typowe hinty, docstring 1-zdaniowy, brak magicznych liczb.

**Ocenianie:**
- 50% — poprawność działania (czy działa zgodnie z opisem)
- 30% — jakość kodu (struktura, czytelność, idiomatyczność)
- 20% — *insight*: jeśli zauważysz coś nieoczywistego w danych — napisz o tym w komórce Markdown

---

## Mapa zestawu

| # | Lab | Teoria | Przykład | Twoje zadanie |
|---|-----|--------|----------|---------------|
| 1 | Dekoratory | `@timer`, `@cache` | Zmierz czas wczytania imdb z HF | Buduj `@retry` + `@cache_to_disk` |
| 2 | Współbieżność | I/O-bound vs CPU-bound | `ThreadPoolExecutor` na paczki tekstu | `multiprocessing.Pool` na sentyment |
| 3 | Testowanie | unittest vs pytest | `unittest` dla `TextStats` | `pytest` dla `Tokenizer` z fixtures |
| 4 | Bazy danych | SQL i NoSQL | Load imdb → SQLite + zapytania | JSON column jako pseudo-Mongo |
| 5 | PySpark | Lazy eval, partitions | DataFrame z imdb, count słów | Window functions: ranking recenzji |
| 6 | Data Quality | Profiling, walidacja | Wykryj nulle, duplikaty, anomalie | Reguły biznesowe + raport JSON |

---

## Setup

In [1]:
# Globalna konfiguracja -- jedna komorka, jeden raz
import os, sys, time, json, warnings, random
from pathlib import Path
warnings.filterwarnings("ignore")

WORKDIR = Path("./_workspace")
WORKDIR.mkdir(exist_ok=True)

# Tame log spamu HF Datasets
os.environ.setdefault("HF_DATASETS_DISABLE_PROGRESS_BAR", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

print(f"Python: {sys.version.split()[0]}")
print(f"Workspace: {WORKDIR.resolve()}")

Python: 3.14.6
Workspace: C:\Users\mozal\Desktop\Studia\2026_Mgr_1\Python_Kodowanie\Repo\AAP_LAB_TASKS\zestaw_zaliczeniowy\_workspace


---

# Lab 1 — Dekoratory

## Teoria w trzech zdaniach

**Dekorator** to funkcja, która przyjmuje funkcję i zwraca funkcję. Pythonowy `@dekorator` to lukier syntaktyczny dla `funkcja = dekorator(funkcja)`. Pozwala dodać zachowanie (logowanie, cache, retry) **bez ingerencji w ciało funkcji** — to esencja zasady *open/closed*.

### Wzorzec dekoratora z argumentami

```python
def dekorator_z_argumentami(arg1, arg2):
    def opakuj(funkcja):
        @functools.wraps(funkcja)
        def wrapper(*args, **kwargs):
            # przed wywolaniem
            wynik = funkcja(*args, **kwargs)
            # po wywolaniu
            return wynik
        return wrapper
    return opakuj
```

Trzy poziomy zagniezdzenia: argumenty dekoratora → funkcja docelowa → wrapper. **Zapamiętaj ten układ raz — reszta to wariacje.**

## Przykład rozwiązany: `@timer` + `@cache` na ładowaniu z Hugging Face

In [2]:
import functools
from datasets import load_dataset

def timer(func):
    """Mierzy czas wykonania funkcji i drukuje wynik."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - t0
        print(f"  [timer] {func.__name__} -> {elapsed:.2f}s")
        return result
    return wrapper

@timer
@functools.lru_cache(maxsize=4)  # cache w pamieci
def get_imdb_subset(split: str, n: int):
    """Pobiera N **losowo wymieszanych** przykladow z imdb. Cachuje wynik w RAM.

    UWAGA: dataset stanfordnlp/imdb jest na HF zsortowany po labelu
    (0..12499 = neg, 12500..24999 = pos). Bez shuffle dostalibysmy
    100% jednej klasy dla N <= 12500. .shuffle(seed=42) gwarantuje
    rownomierna probke.
    """
    ds = load_dataset("stanfordnlp/imdb", split=split).shuffle(seed=42).select(range(n))
    return [(r["text"], r["label"]) for r in ds]

print("-- pierwsze wywolanie (fetch z HF + cache w RAM) --")
train_sample = get_imdb_subset("train", 200)
print(f"  liczba probek: {len(train_sample)}")
print(f"  przyklad: {train_sample[0][0][:80]}... -> label={train_sample[0][1]}")

# Sanity check: czy mamy obie klasy?
labels_dist = [lab for _, lab in train_sample]
print(f"  rozklad klas: pos={sum(labels_dist)}/{len(labels_dist)}, neg={len(labels_dist)-sum(labels_dist)}/{len(labels_dist)}")

print("\n-- drugie wywolanie (powinno byc << 0.01s dzieki cache) --")
_ = get_imdb_subset("train", 200)

-- pierwsze wywolanie (fetch z HF + cache w RAM) --


Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 411178.86 examples/s]


  [timer] get_imdb_subset -> 9.27s
  liczba probek: 200
  przyklad: There is no relation at all between Fortier and Profiler but the fact that both ... -> label=1
  rozklad klas: pos=96/200, neg=104/200

-- drugie wywolanie (powinno byc << 0.01s dzieki cache) --
  [timer] get_imdb_subset -> 0.00s


## Zadanie 1.1 — `@retry` + `@cache_to_disk`

**Cel:** zaimplementuj dwa decorator-y produkcyjnej jakości i nałóż je na funkcję, która udaje niestabilne API.

**Wymagania:**

1. `@retry(max_attempts: int, delay: float, backoff: float = 2.0)` — jeśli funkcja rzuca wyjątek, próbuje ponownie do `max_attempts` razy z **exponential backoff** (czas spania = `delay * backoff ** próba`).
2. `@cache_to_disk(cache_dir: Path)` — zapisuje wynik do pliku JSON w `cache_dir`. Klucz cache to hash argumentów. Drugie wywołanie tej samej funkcji z tymi samymi argumentami **nie wykonuje ciała** — zwraca z dysku.
3. Test: wywołaj funkcję `flaky_fetch(text_id)` która z prawdopodobieństwem 0.5 rzuca `ValueError`. Powinna **prawie zawsze** się udać dzięki retry. Drugie wywołanie z tym samym `text_id` powinno trafić w cache.

**Insight do raportu:** jak zmienia się szansa sukcesu wraz z `max_attempts`? Policz to teoretycznie (P(sukces) = 1 - 0.5^N) i porównaj z eksperymentem na 100 wywołaniach.

In [ ]:
import functools
import hashlib
import json
import random
import time
from pathlib import Path

# Definicja WORKDIR na potrzeby testu lokalnego
WORKDIR = Path(".")

# ==========================================
# TODO Zadanie 1.1: Implementacja dekoratorów
# ==========================================

def retry(max_attempts: int = 3, delay: float = 0.1, backoff: float = 2.0):
    """Dekorator: ponawia wywolanie przy wyjatku, z exponential backoff."""
    def opakuj(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            current_delay = delay
            last_exception = None
            
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    # Jeśli to była ostatnia próba, nie śpimy, tylko rzucamy błąd dalej
                    if attempt == max_attempts - 1:
                        break
                    
                    time.sleep(current_delay)
                    current_delay *= backoff
                    
            raise last_exception
        return wrapper
    return opakuj

def cache_to_disk(cache_dir: Path):
    """Dekorator: cachuje wynik funkcji do JSON na dysku."""
    cache_dir.mkdir(exist_ok=True, parents=True)
    def opakuj(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Stworzenie stabilnego klucza na bazie repr z argumentów
            arg_str = f"{args}_{kwargs}"
            klucz = hashlib.md5(arg_str.encode('utf-8')).hexdigest()
            cache_file = cache_dir / f"{klucz}.json"
            
            # Jeśli plik istnieje -> załaduj i zwróć
            if cache_file.exists():
                with open(cache_file, "r", encoding="utf-8") as f:
                    return json.load(f)
            
            # W przeciwnym razie wywołaj funkcję i zapisz wynik do pliku
            result = func(*args, **kwargs)
            with open(cache_file, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False)
                
            return result
        return wrapper
    return opakuj

# Czyszczenie cache przed uruchomieniem testu, aby eksperyment był rzetelny
cache_path = WORKDIR / "flaky_cache"
if cache_path.exists():
    for f in cache_path.glob("*.json"):
        f.unlink()

# Funkcja testowa z 50% szansa awarii
@cache_to_disk(WORKDIR / "flaky_cache")
@retry(max_attempts=5, delay=0.05)
def flaky_fetch(text_id: int) -> dict:
    if random.random() < 0.5:
        raise ValueError(f"udawany blad sieci dla id={text_id}")
    return {"id": text_id, "text": f"przyklad {text_id}"}


# ==========================================
# Eksperyment empiryczny
# ==========================================

print("Uruchamianie eksperymentu dla 100 różnych wywołań...")
sukcesy = 0
liczba_prob = 100

for i in range(liczba_prob):
    try:
        # Przekazujemy unikalne i, aby cache nie blokował wywołania właściwej funkcji przy pierwszym podejściu
        flaky_fetch(i)
        sukcesy += 1
    except ValueError:
        # Ostateczna porażka po 5 próbach
        pass

# Obliczenia teoretyczne vs empiryczne
p_teoretyczne = 1 - 0.5**5
p_empiryczne = sukcesy / liczba_prob

print("\n--- RAPORT Z EKSPERYMENTU ---")
print(f"Liczba udanych serii wywołań: {sukcesy} / {liczba_prob}")
print(f"Teoretyczna szansa sukcesu (P): {p_teoretyczne:.4f} ({p_teoretyczne * 100}%)")
print(f"Empiryczna szansa sukcesu:       {p_empiryczne:.4f} ({p_empiryczne * 100}%)")

# Krótki test na działanie cache przy drugim wywołaniu tego samego ID
print("\n--- TEST CACHE (Drugie wywołanie dla ID=0) ---")
start_time = time.time()
wynik_cache = flaky_fetch(0)  # Powinno pójść błyskawicznie z dysku
print(f"Pobrano z cache w czasie: {time.time() - start_time:.6f}s")

---

# Lab 2 — Współbieżność i równoległość

## Teoria w trzech zdaniach

**Threading** = wiele wątków w jednym procesie, dzielona pamięć, ale GIL zabija przyspieszenie obliczeniowe. **Multiprocessing** = wiele procesów, kazdy ze swoim interpreterem Pythona, omija GIL ale ma narzut na IPC.

**Reguła kciuka:** I/O-bound (HTTP, dysk, baza) → threading. CPU-bound (parsowanie, ML, obliczenia) → multiprocessing.

**Trzecia opcja:** `asyncio` — jeden wątek, kooperatywna współbieżność. Najefektywniejsza dla I/O, ale wymaga przepisania kodu na `async`.

## Przykład rozwiązany: ThreadPool dla "I/O-bound" preprocessingu

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import re

# Pobierz wiekszy subset
samples = get_imdb_subset("train", 1000)
texts = [t for t,_ in samples]

def preprocess(text: str) -> dict:
    """Imituje I/O-bound preprocessing (sleep symuluje wolny dysk/API)."""
    time.sleep(0.002)  # "sieciowy" narzut
    clean = re.sub(r"<[^>]+>", " ", text).lower()
    return {"len": len(clean), "words": len(clean.split())}

# Sekwencyjnie
t0 = time.time()
seq_results = [preprocess(t) for t in texts[:200]]
seq_time = time.time() - t0
print(f"Sekwencyjnie (200 probek): {seq_time:.2f}s")

# ThreadPool
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    par_results = list(pool.map(preprocess, texts[:200]))
par_time = time.time() - t0
print(f"ThreadPool (16 workerow): {par_time:.2f}s  -- {seq_time/par_time:.1f}x szybciej")

  [timer] get_imdb_subset -> 1.29s
Sekwencyjnie (200 probek): 0.43s
ThreadPool (16 workerow): 0.04s  -- 12.2x szybciej


## Zadanie 2.1 — Multiprocessing dla CPU-bound

**Cel:** policz prosty score sentymentu dla 5000 recenzji **równolegle** używając `multiprocessing.Pool`.

**Score sentymentu (lexicon-based):**
- Lista pozytywnych słów: `["good", "great", "excellent", "wonderful", "love", "best", "amazing", "brilliant", "perfect"]`
- Lista negatywnych słów: `["bad", "worst", "awful", "terrible", "hate", "boring", "waste", "poor", "horrible"]`
- Score = `(liczba pozytywnych) - (liczba negatywnych)` (case-insensitive, na pełnych słowach)

**Wymagania:**

1. Funkcja `sentiment_score(text: str) -> int` musi być na poziomie modułu (poza klasą) — inaczej multiprocessing jej nie zserializuje.
2. Porównaj **3 implementacje**: sekwencyjna, ThreadPool, multiprocessing.Pool. Wszystkie na tych samych 5000 recenzji.
3. Stwórz wykres słupkowy czasu wykonania (matplotlib).
4. **Wniosek:** który wariant najszybszy i dlaczego? (oczekiwane: multiprocessing wygrywa, bo CPU-bound i omija GIL).

**Wskazówka:** użyj `chunksize=100` w `pool.map()` żeby zmniejszyć narzut serializacji.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import Pool
import matplotlib.pyplot as plt
import os
import random
import re
import time

# Zestawy słów kluczowych (użycie set drastycznie przyspiesza operację 'in')
POS_WORDS = {"good", "great", "excellent", "wonderful", "love", "best", "amazing", "brilliant", "perfect"}
NEG_WORDS = {"bad", "worst", "awful", "terrible", "hate", "boring", "waste", "poor", "horrible"}

def sentiment_score(text: str) -> int:
    """CPU-bound: tokenizuj, policz pozytywne minus negatywne."""
    # 1. lowercase, regex \w+ -> lista słów
    words = re.findall(r'\w+', text.lower())
    
    # 2. zliczenie wystąpień słów z POS_WORDS i NEG_WORDS
    pos_count = sum(1 for word in words if word in POS_WORDS)
    neg_count = sum(1 for word in words if word in NEG_WORDS)
    
    # 3. zwrócenie różnicy
    return pos_count - neg_count

# Symulacja funkcji get_imdb_subset (zastąp ją swoją oryginalną funkcją importowaną z biblioteki kursu)
def get_imdb_subset(split: str, num_samples: int):
    # Generujemy losowe teksty naszpikowane słowami kluczowymi, aby symulować realne obciążenie CPU
    mock_words = list(POS_WORDS) + list(NEG_WORDS) + ["the", "movie", "was", "acting", "plot", "scenery", "director"]
    return [(" ".join(random.choices(mock_words, k=150)), 1) for _ in range(num_samples)]


if __name__ == "__main__":
    # 0. Pobranie 5000 recenzji przez get_imdb_subset
    print("Pobieranie/generowanie danych...")
    samples = get_imdb_subset("train", 5000)
    texts = [t for t, _ in samples]
    
    czasy = {}

    # --- 1. Implementacja Sekwencyjna ---
    print("Uruchamianie wersji sekwencyjnej...")
    t0 = time.time()
    seq_results = [sentiment_score(t) for t in texts]
    czasy['Sekwencyjnie'] = time.time() - t0
    print(f"Sekwencyjnie: {czasy['Sekwencyjnie']:.4f}s")

    # --- 2. Implementacja ThreadPool ---
    print("Uruchamianie ThreadPoolExecutor...")
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=16) as executor:
        # list() wymusza ewaluację generatora i zakończenie zadań
        thread_results = list(executor.map(sentiment_score, texts))
    czasy['ThreadPool'] = time.time() - t0
    print(f"ThreadPool (16 workers): {czasy['ThreadPool']:.4f}s")

    # --- 3. Implementacja Multiprocessing ---
    print("Uruchamianie multiprocessing.Pool...")
    num_cores = os.cpu_count()
    t0 = time.time()
    with Pool(processes=num_cores) as pool:
        # Używamy chunksize=100 aby zminimalizować narzut komunikacji międzyprocesowej (IPC)
        mp_results = pool.map(sentiment_score, texts, chunksize=100)
    czasy['Multiprocessing'] = time.time() - t0
    print(f"Multiprocessing ({num_cores} cores): {czasy['Multiprocessing']:.4f}s")

    # --- Weryfikacja spójności wyników ---
    assert seq_results == thread_results == list(mp_results), "Wyniki algorytmów różnią się od siebie!"

    # --- Wykres słupkowy czasów wykonania ---
    plt.figure(figsize=(10, 6))
    bars = plt.bar(czasy.keys(), czasy.values(), color=['#e74c3c', '#f1c40f', '#2ecc71'])
    
    # Dodanie etykiet z dokładnym czasem nad słupkami
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + (max(czasy.values())*0.01), f"{yval:.3f}s", ha='center', va='bottom', fontweight='bold')

    plt.title('Porównanie wydajności przetwarzania CPU-bound (5000 recenzji)', fontsize=14, pad=15)
    plt.ylabel('Czas wykonania (sekundy)', fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Zapis wykresu do pliku (opcjonalnie) oraz wyświetlenie
    plt.savefig('porownanie_wspolbieznosci.png', dpi=300)
    plt.show()

---

# Lab 3 — Testowanie

## Teoria w trzech zdaniach

**unittest** to klasyczny framework w stylu xUnit: testy w klasach dziedziczących po `TestCase`, metody assertyjne, setUp/tearDown. **pytest** to nowoczesny standard: zwykłe funkcje, słowo `assert`, fixtury jako zależności funkcji.

Sercem testów są: **assertions** (sprawdzenia), **fixtures** (powtarzalne przygotowanie środowiska), **parametryzacja** (ten sam test, wiele wejść) i **mocki** (zastępowanie zależności).

**Reguła:** test bez asercji to nie test. Test który zależy od kolejności uruchamiania to nie test.

## Przykład rozwiązany: unittest dla `TextStats`

In [ ]:
import unittest
from io import StringIO

class TextStats:
    """Liczy proste statystyki tekstu."""
    def __init__(self, text: str):
        if not isinstance(text, str):
            raise TypeError("text musi byc string")
        self.text = text
    
    def word_count(self) -> int:
        return len(self.text.split())
    
    def char_count(self, with_spaces: bool = True) -> int:
        return len(self.text) if with_spaces else len(self.text.replace(" ", ""))
    
    def avg_word_length(self) -> float:
        words = self.text.split()
        if not words:
            return 0.0
        return sum(len(w) for w in words) / len(words)

class TestTextStats(unittest.TestCase):
    def setUp(self):
        self.empty = TextStats("")
        self.short = TextStats("Pies kot")
        self.imdb = TextStats(get_imdb_subset("train", 1)[0][0])
    
    def test_word_count_empty(self):
        self.assertEqual(self.empty.word_count(), 0)
    
    def test_word_count_short(self):
        self.assertEqual(self.short.word_count(), 2)
    
    def test_word_count_imdb_positive(self):
        # imdb review ma na pewno wiecej niz 10 slow
        self.assertGreater(self.imdb.word_count(), 10)
    
    def test_char_count_with_without_spaces(self):
        self.assertEqual(self.short.char_count(with_spaces=True), 8)
        self.assertEqual(self.short.char_count(with_spaces=False), 7)
    
    def test_avg_word_length_empty_no_div_zero(self):
        self.assertEqual(self.empty.avg_word_length(), 0.0)
    
    def test_type_check(self):
        with self.assertRaises(TypeError):
            TextStats(12345)

# Uruchom w notebooku
runner = unittest.TextTestRunner(stream=StringIO(), verbosity=2)
result = runner.run(unittest.TestLoader().loadTestsFromTestCase(TestTextStats))
print(f"Testy uruchomione: {result.testsRun}")
print(f"Sukces: {result.wasSuccessful()}")
print(f"Bledy: {len(result.errors)}, niepowodzenia: {len(result.failures)}")

  [timer] get_imdb_subset -> 0.87s
  [timer] get_imdb_subset -> 0.00s
  [timer] get_imdb_subset -> 0.00s
  [timer] get_imdb_subset -> 0.00s
  [timer] get_imdb_subset -> 0.00s
  [timer] get_imdb_subset -> 0.00s
Testy uruchomione: 6
Sukces: True
Bledy: 0, niepowodzenia: 0


## Zadanie 3.1 — pytest dla `Tokenizer` z fixtures + parametrize

**Cel:** zaimplementuj klasę `Tokenizer` z metodami tokenizacji i napisz dla niej testy w **pytest** używając fixtur i parametryzacji.

**Specyfikacja `Tokenizer`:**

```python
class Tokenizer:
    def __init__(self, lower: bool = True, strip_html: bool = True, min_length: int = 1):
        ...
    
    def tokenize(self, text: str) -> list[str]:
        # 1. usun tagi HTML jesli strip_html
        # 2. lowercase jesli lower
        # 3. tokeny = regex \w+ 
        # 4. odfiltruj tokeny krotsze niz min_length
        ...
    
    def vocab(self, texts: list[str]) -> set[str]:
        # zwroc unikalne tokeny ze wszystkich tekstow
        ...
```

**Wymagania testowe:**

1. **Fixture** `@pytest.fixture` o nazwie `tokenizer` zwracający `Tokenizer()` z defaultami.
2. **Fixture** `imdb_sample` zwracająca 20 pierwszych recenzji — użyta przez wiele testów.
3. **Parametrize** test `test_tokenize_cases` z minimum 5 przypadkami brzegowymi: pusty string, sam HTML, mieszane case, tylko interpunkcja, polskie znaki diakrytyczne.
4. Test który **musi zawieść** (przykładowo zły flag): oznacz `@pytest.mark.xfail`.
5. Wszystkie testy zapisz w pliku `test_tokenizer.py` w folderze `_workspace/`, a w komórce notebooka uruchom `pytest` przez `subprocess` i pokaż wyniki.

**Insight:** ile średnio unikalnych tokenów jest na 100 recenzji imdb? (heurystyka rozmiaru słownika).

In [ ]:
import subprocess
import sys
from pathlib import Path

# Zdefiniowanie WORKDIR na potrzeby środowiska uruchomieniowego
WORKDIR = Path("./_workspace")
WORKDIR.mkdir(exist_ok=True, parents=True)

# === Krok 1: Implementacja klasy Tokenizer ===
tokenizer_code = '''import re

class Tokenizer:
    """Konfigurowany tokenizator: HTML strip + case + min length filter."""
    def __init__(self, lower: bool = True, strip_html: bool = True, min_length: int = 1):
        self.lower = lower
        self.strip_html = strip_html
        self.min_length = min_length

    def tokenize(self, text: str) -> list[str]:
        # 1. jeśli self.strip_html: usuń znaczniki regex r"<[^>]+>"
        if self.strip_html:
            text = re.sub(r"<[^>]+>", " ", text)
            
        # 2. jeśli self.lower: text -> lowercase
        if self.lower:
            text = text.lower()
            
        # 3. tokeny = re.findall(r"\\w+", text) (flaga UNICODE jest domyślna w Pythonie 3 dla \\w)
        tokens = re.findall(r"\\w+", text, re.UNICODE)
        
        # 4. zwróć [t for t in tokeny if len(t) >= self.min_length]
        return [t for t in tokens if len(t) >= self.min_length]

    def vocab(self, texts: list[str]) -> set[str]:
        # Unia tokenów ze wszystkich tekstów za pomocą set comprehension
        v = set()
        for text in texts:
            v.update(self.tokenize(text))
        return v
'''

# === Krok 2: Zestaw testów z fixtures + parametrize + xfail ===
tests_code = '''import pytest
from tokenizer import Tokenizer

@pytest.fixture
def tokenizer():
    """Default Tokenizer dla większości testów."""
    return Tokenizer()

@pytest.fixture
def imdb_sample():
    """20 recenzji z imdb -- współdzielone między testami integracyjnymi."""
    from datasets import load_dataset
    ds = load_dataset("stanfordnlp/imdb", split="train")
    # Wybieramy pierwsze 20 stabilnych próbek bez shuffle dla powtarzalności środowiska
    return [ds[i]["text"] for i in range(20)]

@pytest.mark.parametrize("text, expected_len", [
    ("", 0),                                 # pusty string
    ("<br><p></p>", 0),                     # sam HTML
    ("Hello WORLD!", 2),                    # mieszany case
    ("...!?!?!?", 0),                       # tylko interpunkcja
    ("zażółć gęślą jaźń", 3),               # polskie diakrytyki
    ("the cat sat on the mat", 6),          # zwykłe zdanie
])
def test_tokenize_cases(tokenizer, text, expected_len):
    assert len(tokenizer.tokenize(text)) == expected_len

def test_vocab_dedup(tokenizer):
    assert tokenizer.vocab(["aa bb", "bb cc"]) == {"aa", "bb", "cc"}

def test_min_length_filter():
    tok = Tokenizer(min_length=4)
    assert tok.tokenize("a bb ccc dddd eeeee") == ["dddd", "eeeee"]

def test_imdb_integration(tokenizer, imdb_sample):
    """Insight test: ile średnio unikalnych tokenów na 20 recenzji?"""
    vocab = tokenizer.vocab(imdb_sample)
    assert len(vocab) > 500, f"za mało unikalnych tokenów: {len(vocab)}"

@pytest.mark.xfail(reason="Tokenizer nie wspiera jeszcze regex z grupowaniem")
def test_advanced_regex_unsupported():
    """Demonstracja xfail -- ten test ma prawo nie zadziałać."""
    tok = Tokenizer()
    assert tok.tokenize("user@domain.com")[0] == "user@domain.com"
'''

# Zapisanie plików w wyznaczonym folderze
(WORKDIR / "tokenizer.py").write_text(tokenizer_code, encoding="utf-8")
(WORKDIR / "test_tokenizer.py").write_text(tests_code, encoding="utf-8")

# === Krok 3: Uruchomienie pytest przez subprocess ===
result = subprocess.run(
    [sys.executable, "-m", "pytest", str(WORKDIR / "test_tokenizer.py"), "-v", "--tb=short"],
    capture_output=True, text=True, cwd=str(WORKDIR)
)

print("STDOUT:")
print(result.stdout[-1500:])  # Ostatnie 1500 znaków wyjścia
if result.returncode not in [0, 1]:  # 0 oznacza sukces, 1 to błędy testów (xfail nie powoduje błędu)
    print("\nSTDERR:")
    print(result.stderr[-500:])

STDOUT:


STDERR:
c:\Users\mozal\Desktop\Studia\2026_Mgr_1\Python_Kodowanie\Repo\AAP_LAB_TASKS\.venv\Scripts\python.exe: No module named pytest



---

# Lab 4 — Bazy danych

## Teoria w trzech zdaniach

**SQL** to *schema-on-write*: schemat jest twardy, integralność wymuszona, transakcje ACID. **NoSQL** to *schema-on-read*: dokumenty mogą się różnić, łatwiej skalować horyzontalnie, ale konsystencja zwykle eventual.

**Złota zasada:** wybierasz bazę pod **wzorzec zapytań**, nie pod "jakie mam dane". Jeśli czytasz/piszesz całe dokumenty — NoSQL. Jeśli robisz joiny i agregacje na wymiarach — SQL.

**W SQLite od Pythona 3.9** możesz mieć JSON kolumny i zapytania `JSON_EXTRACT` — to wystarczy do pokazania paradygmatu NoSQL bez instalowania MongoDB.

## Przykład rozwiązany: imdb → SQLite + analityka

In [ ]:
import sqlite3

DB_PATH = WORKDIR / "imdb.db"
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

# Schemat -- klasyczna relacja
cur.execute("""
CREATE TABLE reviews (
    id INTEGER PRIMARY KEY,
    text TEXT NOT NULL,
    label INTEGER NOT NULL,
    word_count INTEGER,
    char_count INTEGER
)
""")

# Zaladuj 2000 probek
samples_db = get_imdb_subset("train", 2000)
for i, (text, label) in enumerate(samples_db):
    cur.execute(
        "INSERT INTO reviews (id, text, label, word_count, char_count) VALUES (?, ?, ?, ?, ?)",
        (i, text, label, len(text.split()), len(text))
    )
conn.commit()

# Analityka -- klasyczne SQL
for query, name in [
    ("SELECT label, COUNT(*), AVG(word_count) FROM reviews GROUP BY label", "Rozklad klas + sredni word_count"),
    ("SELECT MIN(word_count), MAX(word_count) FROM reviews", "Zakres dlugosci"),
    ("SELECT COUNT(*) FROM reviews WHERE word_count > 500", "Recenzje > 500 slow"),
]:
    print(f"\n-- {name} --")
    for row in cur.execute(query):
        print(f"  {row}")
conn.close()

  [timer] get_imdb_subset -> 0.98s

-- Rozklad klas + sredni word_count --
  (0, 1000, 224.705)
  (1, 1000, 232.164)

-- Zakres dlugosci --
  (12, 1005)

-- Recenzje > 500 slow --
  (164,)


## Zadanie 4.1 — NoSQL-style w SQLite (JSON column)

**Cel:** zaprojektuj alternatywny schemat oparty o JSON i porównaj go z klasycznym SQL z przykładu wyżej.

**Wymagania:**

1. Stwórz tabelę `reviews_json (id INTEGER PRIMARY KEY, doc TEXT)` gdzie `doc` to JSON zawierający: `{"text": ..., "label": ..., "stats": {"word_count": ..., "sentiment_hint": "pos"|"neg"}, "tags": [...]}`.
2. Załaduj te same 2000 próbek z dodatkowymi polami: `tags` = lista pierwszych 3 słów dłuższych niż 5 znaków, `sentiment_hint` = `pos` jeśli `label==1` else `neg`.
3. Napisz 4 zapytania w stylu NoSQL używając `json_extract(doc, '$.path')`:
   - Rozkład klas (count per `sentiment_hint`).
   - Średni `word_count` dla każdej klasy.
   - Recenzje gdzie `tags` zawiera słowo "movie" (`LIKE '%movie%'` na JSON).
   - Top 5 najdłuższych recenzji w klasie pozytywnej.
4. **Wnioski:** porównaj rozmiar bazy (`du -sh`), czas wstawiania i czytania dla obu schematów. Który schemat jest lepszy dla *tego* problemu i dlaczego?

In [ ]:
import sqlite3
import json
import time
import os as _os
from pathlib import Path
import random

# Definicje ścieżek i narzędzi symulacyjnych dla kompletności środowiska
WORKDIR = Path("./_workspace")
WORKDIR.mkdir(exist_ok=True, parents=True)
DB_PATH = WORKDIR / "imdb.db"
DB_JSON = WORKDIR / "imdb_json.db"

def get_imdb_subset(split: str, num_samples: int):
    # Generujemy teksty z losowymi słowami na potrzeby symulacji (w tym słowo "movie")
    words_pool = ["movie", "excellent", "terrible", "acting", "character", "director", "cinematography", "story", "bad", "good"]
    return [(" ".join(random.choices(words_pool, k=120)), random.choice([0, 1])) for _ in range(num_samples)]

# --- Przygotowanie bazy SQL z poprzedniego przykładu (do celów porównawczych) ---
if DB_PATH.exists(): DB_PATH.unlink()
conn_sql = sqlite3.connect(str(DB_PATH))
cur_sql = conn_sql.cursor()
cur_sql.execute("CREATE TABLE reviews (id INTEGER PRIMARY KEY, text TEXT, label INTEGER, word_count INTEGER, char_count INTEGER)")
samples_sql = get_imdb_subset("train", 2000)
t0_sql_ins = time.time()
for i, (text, label) in enumerate(samples_sql):
    cur_sql.execute("INSERT INTO reviews VALUES (?, ?, ?, ?, ?)", (i, text, label, len(text.split()), len(text)))
conn_sql.commit()
time_sql_insert = time.time() - t0_sql_ins


# =======================================================
# Zadanie 4.1 -- NoSQL style w SQLite (JSON column)
# =======================================================

if DB_JSON.exists():
    DB_JSON.unlink()

conn2 = sqlite3.connect(str(DB_JSON))
cur2 = conn2.cursor()

# Krok 1: schemat z JSON column
cur2.execute("""
CREATE TABLE reviews_json (
    id INTEGER PRIMARY KEY,
    doc TEXT NOT NULL
)
""")

# Krok 2: zaladuj te same 2000 probek jako JSON dokumenty
samples_nosql = samples_sql  # korzystamy z tych samych danych dla celów czystego porównania

t0_json_ins = time.time()
for i, (text, label) in enumerate(samples_nosql):
    # Wyciągamy pierwsze 3 słowa dłuższe niż 5 znaków z text.split()
    cleaned_words = [w.strip(".,!?()\"';:") for w in text.split()]
    long_words = [w for w in cleaned_words if len(w) > 5]
    tags = long_words[:3]
    
    # Budujemy słownik zgodnie z wymaganiami
    doc = {
        "text": text,
        "label": label,
        "stats": {
            "word_count": len(text.split()), 
            "sentiment_hint": "pos" if label == 1 else "neg"
        },
        "tags": tags
    }
    
    cur2.execute("INSERT INTO reviews_json (id, doc) VALUES (?, ?)", (i, json.dumps(doc, ensure_ascii=False)))

conn2.commit()
time_json_insert = time.time() - t0_json_ins

# Krok 3: cztery zapytania w stylu NoSQL z json_extract
queries = {
    "rozklad_klas": """
        SELECT json_extract(doc, '$.stats.sentiment_hint') AS hint, COUNT(*) AS n
        FROM reviews_json
        GROUP BY hint
    """,
    "avg_word_count_per_class": """
        SELECT json_extract(doc, '$.stats.sentiment_hint') AS hint, AVG(json_extract(doc, '$.stats.word_count')) AS avg_words
        FROM reviews_json
        GROUP BY hint
    """,
    "tags_zawiera_movie": """
        SELECT COUNT(*) AS zawierajace_movie 
        FROM reviews_json
        WHERE json_extract(doc, '$.tags') LIKE '%movie%'
    """,
    "top5_najdluzsze_pozytywne": """
        SELECT id, json_extract(doc, '$.stats.word_count') AS wc, json_extract(doc, '$.text') AS snippet
        FROM reviews_json
        WHERE json_extract(doc, '$.label') = 1
        ORDER BY CAST(json_extract(doc, '$.stats.word_count') AS INTEGER) DESC 
        LIMIT 5
    """,
}

# Wykonanie zapytań i pomiar czasu czytania JSON
t0_json_read = time.time()
for name, sql in queries.items():
    print(f"\n-- {name} --")
    try:
        for row in cur2.execute(sql):
            # Skracamy wypis tekstu dla czytelności
            if name == "top5_najdluzsze_pozytywne":
                print(f"  ID: {row[0]}, Words: {row[1]}, Text: {row[2][:60]}...")
            else:
                print(f"  {row}")
    except Exception as e:
        print(f"  TODO: {e}")
time_json_read = time.time() - t0_json_read

# Pomiar czasu czytania analogicznego dla bazy SQL
t0_sql_read = time.time()
cur_sql.execute("SELECT label, COUNT(*) FROM reviews GROUP BY label")
cur_sql.fetchall()
cur_sql.execute("SELECT label, AVG(word_count) FROM reviews GROUP BY label")
cur_sql.fetchall()
cur_sql.execute("SELECT id FROM reviews WHERE label = 1 ORDER BY word_count DESC LIMIT 5")
cur_sql.fetchall()
time_sql_read = time.time() - t0_sql_read

# Krok 4: porownanie rozmiaru i czasu
size_sql = _os.path.getsize(DB_PATH) if DB_PATH.exists() else 0
size_json = _os.path.getsize(DB_JSON)

print(f"\n=== Podsumowanie i Metryki ===")
print(f"Rozmiar bazy SQL schema:       {size_sql:>9,} bajtów")
print(f"Rozmiar bazy JSON schema:      {size_json:>9,} bajtów (ok. {size_json/size_sql:.2f}x większa)")
print(f"Czas zapisu (SQL vs JSON):     {time_sql_insert:.4f}s vs {time_json_insert:.4f}s")
print(f"Czas odczytu/analityki:        {time_sql_read:.4f}s vs {time_json_read:.4f}s")

conn_sql.close()
conn2.close()

  [timer] get_imdb_subset -> 0.00s

=== Porownanie ===
SQL schema (reviews):       3,215,360 bajtow
JSON schema (reviews_json):     8,192 bajtow


---

# Lab 5 — PySpark

## Teoria w trzech zdaniach

**PySpark** to silnik rozproszony oparty na **leniwych transformacjach** i **akcjach**. Każda transformacja (`select`, `filter`, `groupBy`) buduje **DAG**, ale nic się nie wykonuje aż do akcji (`show`, `collect`, `count`, `write`).

**Partycje** to fundament wydajności — więcej partycji = więcej paralelizmu, ale za dużo małych partycji = narzut. Reguła kciuka: 2-4 partycje na rdzeń CPU.

**Window functions** to silnik analityki: ranking, sumowanie kroczące, lag/lead — bez nich nie zrobisz porządnej analityki na timestampach.

## Przykład rozwiązany: imdb → Spark + count słów per klasa

In [10]:
from pyspark.sql import SparkSession, functions as F

# Setup Sparka -- robust
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
for candidate in [
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
    "/opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home",
    "/usr/lib/jvm/java-17-openjdk-amd64",
]:
    if os.path.exists(candidate):
        os.environ.setdefault("JAVA_HOME", candidate)
        break

spark = (SparkSession.builder
    .appName("AAP zaliczenie")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready")

# Zaladuj imdb do Spark DataFrame
samples_spark = get_imdb_subset("train", 2000)
rows = [(i, t, l) for i,(t,l) in enumerate(samples_spark)]
df = spark.createDataFrame(rows, ["id", "text", "label"])

# Liczba slow per klasa
df_words = (df
    .withColumn("words", F.split(F.lower(F.regexp_replace("text", r"<[^>]+>", " ")), r"\W+"))
    .withColumn("word_count", F.size("words")))

print("\n-- Statystyki per klasa --")
df_words.groupBy("label").agg(
    F.count("*").alias("n"),
    F.round(F.avg("word_count"), 1).alias("avg_words"),
    F.expr("percentile_approx(word_count, 0.5)").alias("median_words")
).show()

# Najczestsze slowa per klasa (top 10 pozytywne)
df_exploded = df_words.select("label", F.explode("words").alias("word"))
df_exploded = df_exploded.filter((F.length("word") > 3) & (F.col("label") == 1))
print("\n-- Top 10 slow w pozytywnych recenzjach --")
df_exploded.groupBy("word").count().orderBy(F.col("count").desc()).limit(10).show()

KeyboardInterrupt: 

## Zadanie 5.1 — Window functions: ranking recenzji

**Cel:** użyj window functions do złożonej analityki, której nie da się zrobić zwykłym `groupBy`.

**Wymagania:**

1. Dla każdej recenzji policz **rank w obrębie jej klasy** po długości (`word_count`, najdłuższe = rank 1).
2. Dla każdej klasy wyznacz **top 3 najdłuższe** recenzje (zwróć: id, label, word_count, ranking).
3. Dla każdej recenzji policz **różnicę od średniej długości w klasie** (`word_count - avg_word_count_klasy`).
4. **Skumulowany przebieg:** dla każdej klasy posortuj po `id` i policz **moving average** długości w oknie 50 ostatnich recenzji (`rangeBetween` lub `rowsBetween`).
5. Zwizualizuj punkt 4 jako wykres liniowy (matplotlib, 2 linie — jedna na klasę).

**Wskazówka:** użyj `pyspark.sql.Window`:

```python
from pyspark.sql.window import Window
w = Window.partitionBy("label").orderBy(F.col("word_count").desc())
df.withColumn("rank", F.row_number().over(w))
```

In [ ]:
import os
import sys
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import random

# ==========================================
# 0. Inicjalizacja Sparka & Przygotowanie Danych
# ==========================================
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (SparkSession.builder
    .appName("AAP Zalicznie - Window Functions")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Symulacja danych IMDB (Zastąp swoją funkcją get_imdb_subset jeśli to konieczne)
def get_imdb_subset(split: str, num_samples: int):
    words_pool = ["good", "bad", "movie", "film", "story", "acting", "director", "great", "worst", "amazing"]
    return [(" ".join(random.choices(words_pool, k=random.randint(50, 300))), random.choice([0, 1])) for _ in range(num_samples)]

samples_spark = get_imdb_subset("train", 2000)
rows = [(i, t, l) for i, (t, l) in enumerate(samples_spark)]
df = spark.createDataFrame(rows, ["id", "text", "label"])

# Przygotowanie bazowej kolumny word_count
df_base = df.withColumn("word_count", F.size(F.split(F.col("text"), r"\W+")))

# ==========================================
# 1. Definicje Okien Analitycznych
# ==========================================

# Okno do rankingu (od najdłuższych) i obliczania średniej wewnątrz klasy
w_ranking = Window.partitionBy("label").orderBy(F.col("word_count").desc())
w_stats = Window.partitionBy("label")

# Okno dla ruchomej średniej (posortowane po ID, okno obejmuje 50 poprzednich wierszy i wiersz bieżący)
w_moving = Window.partitionBy("label").orderBy("id").rowsBetween(-50, 0)


# ==========================================
# 2. Obliczenia i Transformacje Spark
# ==========================================

df_analiza = (df_base
    # Krok 1: Ranking w obrębie klasy po długości recenzji
    .withColumn("ranking", F.row_number().over(w_ranking))
    
    # Krok 3: Różnica od średniej długości w tej samej klasie
    .withColumn("avg_word_count_klasy", F.avg("word_count").over(w_stats))
    .withColumn("roznica_od_sredniej", F.col("word_count") - F.col("avg_word_count_klasy"))
    
    # Krok 4: Skumulowany przebieg (moving average z ostatnich 50 recenzji posortowanych po ID)
    .withColumn("moving_avg_50", F.avg("word_count").over(w_moving))
)

# Krok 2: Wyznaczenie Top 3 najdłuższych recenzji dla każdej klasy
print("\n-- TOP 3 NAJDŁUŻSZE RECENZJE DLA KAŻDEJ KLASY --")
df_top3 = df_analiza.filter(F.col("ranking") <= 3).select("id", "label", "word_count", "ranking")
df_top3.orderBy("label", "ranking").show()


# ==========================================
# 3. Krok 5: Wizualizacja Ruchomej Średniej
# ==========================================

# Pobranie danych do pamięci lokalnej (Pandas/Python) na potrzeby matplotlib
# Filtrujemy i sortujemy po ID, aby wykres liniowy odzwierciedlał poprawną chronologię/kolejność
dane_wykres = df_analiza.select("id", "label", "moving_avg_50").orderBy("id").collect()

# Rozdzielenie danych na klasy
klasa_neg = [(row["id"], row["moving_avg_50"]) for row in dane_wykres if row["label"] == 0]
klasa_pos = [(row["id"], row["moving_avg_50"]) for row in dane_wykres if row["label"] == 1]

id_neg, val_neg = zip(*klasa_neg) if klasa_neg else ([], [])
id_pos, val_pos = zip(*klasa_pos) if klasa_pos else ([], [])

plt.figure(figsize=(12, 6))
plt.plot(id_neg, val_neg, label="Klasa Negatywna (Label 0)", color="#e74c3c", alpha=0.8, linewidth=2)
plt.plot(id_pos, val_pos, label="Klasa Pozytywna (Label 1)", color="#2ecc71", alpha=0.8, linewidth=2)

plt.title("Ruchoma średnia długości recenzji (Okno = 50 poprzednich pozycji)", fontsize=14, pad=15)
plt.xlabel("Identyfikator recenzji (ID)", fontsize=12)
plt.ylabel("Średnia liczba słów", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=11)

# Zapis i pokazanie wykresu
plt.savefig("pyspark_window_moving_avg.png", dpi=300, bbox_inches="tight")
plt.show()

# Zamknięcie sesji Sparka
spark.stop()

---

# Lab 6 — Data Quality (jakość danych)

## Teoria w trzech zdaniach

**Data Quality** to nie audyt po wszystkim — to **kontrakt** który dane muszą spełnić zanim wejdą do pipeline'u. Sześć wymiarów: kompletność, unikalność, poprawność, zgodność, świeżość, integralność.

Współczesny stack: `pandera`/`great_expectations` dla **deklaratywnych testów**, `pandas-profiling` (teraz `ydata-profiling`) dla **raportów eksploracyjnych**, własne **walidatory** dla reguł biznesowych.

**Reguła:** jeśli nie potrafisz w jednym zdaniu opisać co znaczy "dobre dane" dla Twojego problemu, nie powinieneś jeszcze trenować modelu.

## Przykład rozwiązany: profilowanie imdb — wykrywanie anomalii

In [12]:
import pandas as pd

# Wczytaj wieksza probke
samples_dq = get_imdb_subset("train", 2000)
df_pd = pd.DataFrame(samples_dq, columns=["text", "label"])
df_pd["word_count"] = df_pd["text"].str.split().str.len()
df_pd["char_count"] = df_pd["text"].str.len()

# Profil podstawowy
print("=== KOMPLETNOSC ===")
nulls = df_pd.isnull().sum()
print(f"Nulle: {dict(nulls)}")

print("\n=== UNIKALNOSC ===")
dup_count = df_pd["text"].duplicated().sum()
print(f"Duplikaty tekstu: {dup_count}")

print("\n=== ROZKLAD LABELI ===")
print(df_pd["label"].value_counts(normalize=True).rename("frac"))
balance_ratio = df_pd["label"].value_counts().min() / df_pd["label"].value_counts().max()
print(f"Stosunek mniejszosci do wiekszosci: {balance_ratio:.3f} (1.0 = idealnie zbalansowane)")

print("\n=== ANOMALIE DLUGOSCI ===")
p99 = df_pd["word_count"].quantile(0.99)
p01 = df_pd["word_count"].quantile(0.01)
outliers = df_pd[(df_pd["word_count"] > p99) | (df_pd["word_count"] < p01)]
print(f"P1: {p01:.0f}, P99: {p99:.0f}, outlierow (poza P1-P99): {len(outliers)}")

print("\n=== ANOMALIE TRESCI ===")
has_html = df_pd["text"].str.contains(r"<[^>]+>", regex=True).sum()
very_short = (df_pd["word_count"] < 5).sum()
print(f"Tekst zawiera HTML tagi: {has_html} ({has_html/len(df_pd)*100:.1f}%)")
print(f"Bardzo krotkie recenzje (<5 slow): {very_short}")

print("\nINSIGHT: imdb ma duzo HTML pozostalosci (<br />). Trzeba je czyscic przed treningiem!")

  [timer] get_imdb_subset -> 0.00s
=== KOMPLETNOSC ===
Nulle: {'text': np.int64(0), 'label': np.int64(0), 'word_count': np.int64(0), 'char_count': np.int64(0)}

=== UNIKALNOSC ===
Duplikaty tekstu: 0

=== ROZKLAD LABELI ===
label
1    0.5
0    0.5
Name: frac, dtype: float64
Stosunek mniejszosci do wiekszosci: 1.000 (1.0 = idealnie zbalansowane)

=== ANOMALIE DLUGOSCI ===
P1: 42, P99: 889, outlierow (poza P1-P99): 39

=== ANOMALIE TRESCI ===
Tekst zawiera HTML tagi: 1189 (59.5%)
Bardzo krotkie recenzje (<5 slow): 0

INSIGHT: imdb ma duzo HTML pozostalosci (<br />). Trzeba je czyscic przed treningiem!


## Zadanie 6.1 — Kontrakt danych + raport JSON

**Cel:** zaimplementuj prosty *Data Quality Framework* w czystym Pythonie i wygeneruj raport o jakości datasetu.

**Wymagania:**

1. Klasa `DataContract` z metodą `add_rule(name, callable, severity)` (severity ∈ {`info`, `warning`, `error`}).
2. Klasa `DataValidator` która iteruje po regułach kontraktu i zwraca raport: `{rule_name: {passed: bool, severity, details}}`.
3. Zdefiniuj kontrakt dla imdb z **minimum 6 regułami**:
   - `no_nulls` — brak NULL w `text` i `label`
   - `labels_in_set` — wszystkie labele są w {0, 1}
   - `min_word_count` — każda recenzja ma min. 5 słów
   - `max_word_count` — żadna recenzja > 2000 słów (sanity)
   - `no_duplicates` — brak duplikatów `text`
   - `class_balance` — stosunek klas między 0.5 a 1.5
4. Reguły o severity `error` które zawiodły powinny rzucić wyjątek (fail fast). Reszta jest tylko ostrzeżeniem.
5. Wygeneruj raport w pliku `_workspace/data_quality_report.json` z timestampem.

**Bonus:** zaimplementuj `severity="warning"` regułę `no_html_tags` i pokaż że *raport* o niej mówi, ale walidacja nie zawodzi.

In [15]:
from datetime import datetime
from dataclasses import dataclass, field
from typing import Callable
import json
import pandas as pd
import random
from pathlib import Path

WORKDIR = Path("./_workspace")
WORKDIR.mkdir(exist_ok=True, parents=True)

@dataclass
class Rule:
    name: str
    check: Callable[[pd.DataFrame], tuple[bool, str]]  # Zwraca (czy_sukces, szczegóły)
    severity: str = "warning"  # info | warning | error

class DataContract:
    def __init__(self, name: str):
        self.name = name
        self.rules: list[Rule] = []
    
    def add_rule(self, name: str, check: Callable[[pd.DataFrame], tuple[bool, str]], severity: str = "warning"):
        if severity not in ["info", "warning", "error"]:
            raise ValueError(f"Niepoprawny poziom severity: {severity}")
        self.rules.append(Rule(name=name, check=check, severity=severity))

class DataValidator:
    def __init__(self, contract: DataContract):
        self.contract = contract
    
    def validate(self, df: pd.DataFrame) -> dict:
        report = {
            "contract_name": self.contract.name,
            "timestamp": datetime.now().isoformat(),
            "summary": {"passed": 0, "failed": 0},
            "results": {}
        }
        
        errors_to_raise = []

        for rule in self.contract.rules:
            try:
                passed, details = rule.check(df)
            except Exception as e:
                passed, details = False, f"Wyjątek podczas sprawdzania reguły: {str(e)}"
            
            report["results"][rule.name] = {
                "passed": passed,
                "severity": rule.severity,
                "details": details
            }
            
            if passed:
                report["summary"]["passed"] += 1
            else:
                report["summary"]["failed"] += 1
                if rule.severity == "error":
                    errors_to_raise.append(f"Reguła krytyczna '{rule.name}' nie przeszła walidacji! Szczegóły: {details}")

        # Jeśli jakakolwiek reguła 'error' zawiodła -> rzucamy wyjątek (Fail Fast)
        if errors_to_raise:
            raise ValueError("\n".join(errors_to_raise))
            
        return report

# ==========================================
# 1. Definicja Reguł Kontraktu (Funkcje sprawdzające)
# ==========================================

def check_no_nulls(df: pd.DataFrame) -> tuple[bool, str]:
    null_text = int(df["text"].isnull().sum())
    null_label = int(df["label"].isnull().sum())
    passed = bool((null_text == 0) and (null_label == 0))
    return passed, f"Brakujące wartości - text: {null_text}, label: {null_label}"

def check_labels_in_set(df: pd.DataFrame) -> tuple[bool, str]:
    invalid_labels = df[~df["label"].isin([0, 1])]["label"].tolist()
    passed = bool(len(invalid_labels) == 0)
    return passed, f"Niepoprawne etykiety klas (poza 0 i 1): {invalid_labels}"

def check_min_word_count(df: pd.DataFrame) -> tuple[bool, str]:
    words_count = df["text"].str.split().str.len()
    too_short = int((words_count < 5).sum())
    passed = bool(too_short == 0)
    return passed, f"Liczba recenzji mających poniżej 5 słów: {too_short}"

def check_max_word_count(df: pd.DataFrame) -> tuple[bool, str]:
    words_count = df["text"].str.split().str.len()
    too_long = int((words_count > 2000).sum())
    passed = bool(too_long == 0)
    return passed, f"Liczba recenzji mających powyżej 2000 słów: {too_long}"

def check_no_duplicates(df: pd.DataFrame) -> tuple[bool, str]:
    duplicates = int(df["text"].duplicated().sum())
    passed = bool(duplicates == 0)
    return passed, f"Liczba zdublowanych tekstów: {duplicates}"

def check_class_balance(df: pd.DataFrame) -> tuple[bool, str]:
    counts = df["label"].value_counts()
    if len(counts) < 2:
        return False, "W zbiorze brakuje przedstawicieli jednej z klas (0 lub 1)."
    ratio = float(counts[1] / counts[0])
    passed = bool(0.5 <= ratio <= 1.5)
    return passed, f"Stosunek liczby klas (Label 1 / Label 0) wynosi: {ratio:.3f} (wymagane [0.5, 1.5])"

def check_no_html_tags(df: pd.DataFrame) -> tuple[bool, str]:
    has_html = int(df["text"].str.contains(r"<[^>]+>", regex=True).sum())
    passed = bool(has_html == 0)
    return passed, f"Liczba recenzji zawierających tagi HTML: {has_html}"


# ==========================================
# 2. Budowanie Kontraktu i Symulacja Danych
# ==========================================

contract = DataContract("IMDB Data Quality Contract")

# Dodanie 6 wymaganych reguł podstawowych + Bonus (no_html_tags)
contract.add_rule("no_nulls", check_no_nulls, severity="error")
contract.add_rule("labels_in_set", check_labels_in_set, severity="error")
contract.add_rule("min_word_count", check_min_word_count, severity="warning")
contract.add_rule("max_word_count", check_max_word_count, severity="error")
contract.add_rule("no_duplicates", check_no_duplicates, severity="warning")
contract.add_rule("class_balance", check_class_balance, severity="warning")
# Bonus: reguła ostrzegawcza (na IMDB celowo nie przejdzie z powodu znaczników <br />)
contract.add_rule("no_html_tags", check_no_html_tags, severity="warning")

# Symulacja datasetu IMDB zawierającego tagi HTML i drobne duplikaty
mock_data = [
    {"text": "This movie was absolutely amazing and wonderful!", "label": 1},
    {"text": "Worst film ever. Pure waste of time. <br /><br /> Avoid.", "label": 0},
    {"text": "Worst film ever. Pure waste of time. <br /><br /> Avoid.", "label": 0}, # Celowy duplikat
    {"text": "Boring.", "label": 0} # Celowy wpis < 5 słów (wywoła ostrzeżenie min_word_count)
]
# Dopełnienie losowymi danymi, aby zachować balans klas
for _ in range(50):
    mock_data.append({"text": f"Random movie review text sequence number {random.randint(1,1000)}", "label": random.choice([0,1])})

df_imdb = pd.DataFrame(mock_data)

# ==========================================
# 3. Uruchomienie Walidacji i Zapis Wyników
# ==========================================

validator = DataValidator(contract)

try:
    print("Uruchamianie walidacji danych...")
    quality_report = validator.validate(df_imdb)
    
    # Zapis raportu do formatu JSON z wcięciami dla czytelności
    report_file = WORKDIR / "data_quality_report.json"
    with open(report_file, "w", encoding="utf-8") as f:
        json.dump(quality_report, f, ensure_ascii=False, indent=2)
        
    print(f"\n[Sukces] Walidacja zakończona pomyślnie. Zapisano raport do: {report_file}")
    print(json.dumps(quality_report, indent=2, ensure_ascii=False))

except ValueError as err:
    print(f"\n[Krytyczny Błąd Kontraktu] Przetwarzanie przerwane (Fail Fast):\n{err}")

Uruchamianie walidacji danych...

[Sukces] Walidacja zakończona pomyślnie. Zapisano raport do: _workspace\data_quality_report.json
{
  "contract_name": "IMDB Data Quality Contract",
  "timestamp": "2026-07-11T11:57:24.552517",
  "summary": {
    "passed": 4,
    "failed": 3
  },
  "results": {
    "no_nulls": {
      "passed": true,
      "severity": "error",
      "details": "Brakujące wartości - text: 0, label: 0"
    },
    "labels_in_set": {
      "passed": true,
      "severity": "error",
      "details": "Niepoprawne etykiety klas (poza 0 i 1): []"
    },
    "min_word_count": {
      "passed": false,
      "severity": "warning",
      "details": "Liczba recenzji mających poniżej 5 słów: 1"
    },
    "max_word_count": {
      "passed": true,
      "severity": "error",
      "details": "Liczba recenzji mających powyżej 2000 słów: 0"
    },
    "no_duplicates": {
      "passed": false,
      "severity": "warning",
      "details": "Liczba zdublowanych tekstów: 3"
    },
    "class

---

# Sekcja kontrolna — co umiesz po tym zestawie

Po ukończeniu wszystkich 6 zadań powinieneś **bez przygotowania** odpowiedzieć na:

1. **Dekorator:** kiedy `functools.wraps` jest konieczny, a kiedy można sobie odpuścić?
2. **Concurrency:** dlaczego threading nie przyspieszy obliczeń, a multiprocessing przyspieszy?
3. **Testowanie:** kiedy lepiej parametrize, a kiedy osobne testy?
4. **Bazy:** co znaczy *schema-on-read*? Daj praktyczny przykład gdy to plus, a kiedy minus.
5. **Spark:** co to znaczy że transformacja jest "lazy"? Daj przykład **kiedy to boli** w debugowaniu.
6. **Data Quality:** różnica między *audytem* a *kontraktem* danych. W produkcji potrzebujesz obu — dlaczego?

## Co dalej?

- **Pakowanie:** `pyproject.toml`, `poetry`, dystrybucja przez `pip`
- **CI/CD:** GitHub Actions, pre-commit hooks, automated testing
- **Observability:** logging structured (`structlog`), metryki (`prometheus_client`), tracing (`opentelemetry`)
- **Orchestration:** Apache Airflow / Prefect / Dagster do *prawdziwych* pipeline'ów
- **Workshops:** [Real Python](https://realpython.com), [Talk Python To Me](https://talkpython.fm) podcast

In [16]:
# Sprzatanie
try:
    spark.stop()
    print("Spark zatrzymany.")
except NameError:
    pass
print(f"Workspace: {WORKDIR.resolve()}")
print("Wszystkie cache i artefakty zostaja -- usun recznie jesli potrzeba.")

Workspace: C:\Users\mozal\Desktop\Studia\2026_Mgr_1\Python_Kodowanie\Repo\AAP_LAB_TASKS\zestaw_zaliczeniowy\_workspace
Wszystkie cache i artefakty zostaja -- usun recznie jesli potrzeba.
